# extension

## Extending the Hyper-Local Pollution Prototype

This notebook is a workshop extension for the Clean Air Investigator project. The goal is to show how the prototype can become more powerful when station measurements are combined with additional context layers.

We will not claim causality from these extra fields. Instead, we will use them to generate better investigation questions and better evidence-grounded hypotheses.


---
## 1. Extension Data Sources

Possible enrichment layers:
- **Weather**: wind speed, wind direction, temperature, humidity, boundary-layer conditions, rainfall.
- **Satellite**: aerosol optical depth, smoke plume indicators, fire hotspot proximity, land-use patterns.
- **Low-cost sensors**: school, campus, community, or neighborhood PM sensors.
- **Urban activity**: traffic density, construction permits, road closures, festival/firework events, industrial schedules.

These are useful because PM2.5 at one station is a measurement, but explaining why it changed often requires surrounding context.


In [ ]:
import pandas as pd
import plotly.express as px
from IPython.display import display

from clean_air_agent.tools.openaq import find_monitoring_locations, get_air_quality_observations
from clean_air_agent.tools.analytics import compare_stations, find_peak_pollution

locations = find_monitoring_locations(location_name="Delhi", radius_km=10)["locations"]
location_ids = [loc["id"] for loc in locations[:3]]
obs_response = get_air_quality_observations(location_ids, hours=24, parameters=["pm25", "pm10"])

df_observations = pd.DataFrame(obs_response["observations"])
df_observations["timestamp"] = pd.to_datetime(df_observations["timestamp"])
df_observations.head()


---
## 2. Add Workshop-Scale Context Data

For a short workshop, it is enough to start with a small context table. In a production system, these fields could come from weather APIs, satellite datasets, transport feeds, or community sensors.


In [ ]:
context_rows = []
for station in sorted(df_observations["station"].unique()):
    context_rows.append({
        "station": station,
        "weather_wind_speed_kmh": 8 if "Anand" in station else 5,
        "weather_wind_direction": "NW" if "Anand" in station else "W",
        "humidity_pct": 68 if "Anand" in station else 61,
        "traffic_context": "major bus terminal / traffic corridor" if "Anand" in station else "mixed urban road activity",
        "satellite_context": "regional haze visible" if "Anand" in station else "no strong plume signal in sample",
        "sensor_extension_idea": "add school/community sensor within 1 km",
    })

df_context = pd.DataFrame(context_rows)
df_context


---
## 3. Join Observations With Context

This join turns raw measurements into an investigation-ready table. The key design principle is separation of responsibility: deterministic code joins and calculates; Gemini interprets the evidence.


In [ ]:
df_enriched = df_observations.merge(df_context, on="station", how="left")
station_summary = pd.DataFrame(compare_stations(df_enriched))
peak_event = find_peak_pollution(df_enriched, parameter="pm25")

display(station_summary)
print("Peak event:", peak_event)


In [ ]:
fig = px.scatter(
    station_summary,
    x="pm25_mean",
    y="pm25_max",
    color="station",
    size="observation_count",
    title="Station-Level PM2.5 With Enrichment Targets",
    labels={
        "pm25_mean": "Mean PM2.5 (µg/m³)",
        "pm25_max": "Peak PM2.5 (µg/m³)",
        "observation_count": "Observations",
    },
    template="plotly_white",
)
fig.update_layout(height=480)
display(fig)


---
## 4. Prompt Gemini With Evidence, Not Secrets

Use this prompt shape when extending the agent. Notice that the model receives summarized evidence and context, not API keys, raw credentials, or instructions from untrusted data.


In [ ]:
evidence_packet = {
    "station_summary": station_summary.to_dict(orient="records"),
    "peak_event": peak_event,
    "context": df_context.to_dict(orient="records"),
}

prompt = f"""
You are helping investigate hyper-local air pollution.
Use only the evidence below. Do not invent measurements.
Frame explanations as hypotheses, not proven causes.

Evidence:
{evidence_packet}

Return:
1. Observed pattern
2. Possible contributing factors
3. What additional data would improve confidence
4. One practical next step for a prototype
"""

print(prompt[:2000])


---
## 5. Extension Design Challenge

Choose one extension and sketch how it would improve the prototype:

| Extension | Question it helps answer | Example feature |
|---|---|---|
| Weather | Did stagnant air or wind direction align with a spike? | Overlay wind speed and direction near peak PM2.5 hours |
| Satellite | Was there regional haze, smoke, or dust beyond the station network? | Add a satellite-context badge to the investigation report |
| Low-cost sensors | Are station readings representative of nearby schools or streets? | Compare official stations with community sensors |
| Traffic/activity | Did local activity coincide with a pollution increase? | Add a timeline of traffic, construction, or event context |

**Participant task:** Add one new context column to `df_context`, join it to the observations, and update the Gemini prompt so the model can discuss that new evidence safely.


In [ ]:
# Participant workspace: add one extension signal here.
# Example ideas: rainfall_mm, road_construction_nearby, traffic_index, school_sensor_pm25

df_context["your_extension_signal"] = "describe the added signal here"
df_context


---
## 6. Deploy With Google Agent Runtime

For the deployment extension, we use **Google Agent Runtime** instead of a hand-managed web service. Agent Runtime is a fully managed runtime for deploying, operating, and scaling agentic applications. It lets the workshop prototype move from local ADK development into a managed Google Cloud agent environment.

Why this fits the Clean Air Investigator:
- The project already uses **Google ADK**, which has full Agent Runtime support.
- The deployable agent lives in `clean_air_agent/` with `__init__.py` and `agent.py`, matching the expected ADK project shape.
- Agent Runtime deployment produces a managed resource whose API name is `reasoningEngines/...`.
- Participants can focus on the agent logic, tools, data boundaries, and guardrails instead of container operations.

Reference docs:
- https://docs.cloud.google.com/gemini-enterprise-agent-platform/build/runtime
- https://adk.dev/deploy/agent-runtime/deploy/


### 6.1 Set Project and Authenticate

Copy these commands into a terminal. Replace `YOUR_PROJECT_ID` with your Google Cloud project ID.

```bash
export GOOGLE_CLOUD_PROJECT="YOUR_PROJECT_ID"
export GOOGLE_CLOUD_LOCATION="us-central1"
```

```bash
gcloud auth login
```

```bash
gcloud auth application-default login
```

```bash
gcloud config set project "$GOOGLE_CLOUD_PROJECT"
```

Enable the core services used by Agent Runtime deployment:

```bash
gcloud services enable aiplatform.googleapis.com cloudresourcemanager.googleapis.com storage.googleapis.com
```

Workshop note: you need billing enabled and sufficient IAM permissions to enable services and deploy Agent Runtime resources.


### 6.2 Deploy the ADK Agent

The ADK CLI deploys the agent project folder to Agent Runtime with `adk deploy agent_engine`. This packages the ADK agent code and deploys it to the managed runtime.

```bash
adk deploy agent_engine \
  --project="$GOOGLE_CLOUD_PROJECT" \
  --region="$GOOGLE_CLOUD_LOCATION" \
  --display_name="Clean Air Investigator" \
  clean_air_agent
```

If your local ADK version still expects environment variables to be supplied during deployment, use the `.env` file only for workshop experimentation:

```bash
adk deploy agent_engine \
  --project="$GOOGLE_CLOUD_PROJECT" \
  --region="$GOOGLE_CLOUD_LOCATION" \
  --display_name="Clean Air Investigator" \
  --env_file=.env \
  clean_air_agent
```

Production note: avoid shipping raw API keys casually. Prefer managed identity, IAM, and secret-management patterns appropriate for your organization.


### 6.3 Capture the Runtime Resource

A successful deployment prints a resource similar to:

```text
projects/PROJECT_NUMBER/locations/us-central1/reasoningEngines/RESOURCE_ID
```

Record these values for testing:

```bash
export AGENT_RUNTIME_RESOURCE_ID="RESOURCE_ID_FROM_DEPLOY_OUTPUT"
```

Agent Runtime query URL shape:

```text
https://$GOOGLE_CLOUD_LOCATION-aiplatform.googleapis.com/v1/projects/$GOOGLE_CLOUD_PROJECT/locations/$GOOGLE_CLOUD_LOCATION/reasoningEngines/$AGENT_RUNTIME_RESOURCE_ID:query
```

For a workshop demo, it is enough to show the deployed `reasoningEngines/...` resource in the Google Cloud console or Agent Runtime UI, then run one evidence-grounded investigation question against it.


### 6.4 Demo and Cleanup Checklist

Demo the deployed agent by showing:
1. The Agent Runtime deployment resource.
2. One prompt about a location or station.
3. One tool-backed response that cites retrieved/calculated evidence.
4. One guardrail behavior, such as refusing to reveal secrets or avoiding unsupported causality.

After the workshop, delete unused deployed agents from the Agent Runtime UI or with your organization-approved Google Cloud cleanup process to avoid unnecessary cost.


---
## 7. Demo Prompt

In your final demo, show:
1. The station or neighborhood you investigated.
2. The strongest pollution pattern you found.
3. The extra context layer you added.
4. How Gemini interpreted the evidence.
5. One thing your prototype still cannot prove.
